# Fitness Membership — Churn Prediction
## Pipeline sửa hoàn chỉnh: gán nhãn + drop sau split, không leakage

## 0. Import thư viện

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, os

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, make_scorer, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier



RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load dữ liệu

In [2]:
df_raw = pd.read_csv('data_preprocessed.csv')
df_raw['last_visit_date'] = pd.to_datetime(df_raw['last_visit_date'])
df_raw['join_date']       = pd.to_datetime(df_raw['join_date'])

print(f'Shape: {df_raw.shape}')
df_raw.head()

Shape: (1998, 29)


,age,membership_type,visit_per_week,days_per_week,attend_group_lesson,avg_time_check_in,avg_time_check_out,duration_in_gym_minutes,has_drink_subscription,personal_training,uses_sauna,self_identified_gender,subscription_price,subscription_model,adjusted_price,discount_type,discount_rate,final_price,access_hours,home_gym_location,latitude,longitude,join_date,personal_training_hours,multi_location_access,last_visit_date,id,days_since_last_visit,churn
0,40,Premium,2,"Sat, Thu",False,08:39:00,10:37:00,118,False,False,False,Male,50,Monthly,50.0000,Promo,0.0500,47.5000,All hours,"San Jose, CA",37.3382,-121.8863,2024-05-20,0,True,2025-06-30,1,22,1
1,35,Standard,5,"Fri, Sat, Thu, Tue, Wed",True,15:45:00,18:43:00,178,False,False,False,Non-binary,30,Monthly,30.0000,Unkown,0.0000,30.0000,Weekdays only,"Fresno, CA",36.7378,-119.7871,2022-09-14,0,False,2025-05-26,2,57,2
2,39,Standard,4,"Sun, Thu, Tue, Wed",True,13:35:00,14:37:00,62,True,True,True,Male,30,Monthly,30.0000,Loyalty,0.1000,27.0000,Weekdays only,"Oakland, CA",37.8044,-122.2712,2023-05-05,4,False,2025-06-09,3,43,1
3,35,Basic,2,"Mon, Tue",False,17:06:00,18:39:00,93,True,True,False,Male,20,Monthly,20.0000,Unkown,0.0000,20.0000,Off-peak only,"Anaheim, CA",33.8366,-117.9143,2023-09-24,7,False,2025-06-17,4,35,1
4,18,Standard,2,"Sun, Wed",False,14:16:00,16:57:00,161,True,True,False,Prefer not to say,30,Monthly,30.0000,Loyalty,0.1000,27.0000,Weekdays only,"Sacramento, CA",38.5816,-121.4944,2025-01-07,3,True,2025-07-13,5,9,0


## 2. Feature engineering 

In [ ]:
# === FEATURE ENGINEERING NÂNG CAO ===
df = df_raw.copy()

# 1. Từ join_date: giữ nguyên join_season, join_month, join_year
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Fall'

df['join_season'] = df['join_date'].dt.month.apply(get_season)
df['join_month']  = df['join_date'].dt.month
df['join_year']   = df['join_date'].dt.year

# 2. Từ check-in / check-out
df['avg_time_check_in_hour']  = pd.to_datetime(df['avg_time_check_in'], format='%H:%M:%S').dt.hour
df['avg_time_check_out_hour'] = pd.to_datetime(df['avg_time_check_out'], format='%H:%M:%S').dt.hour

# 5. Feature tương tác và engagement
df['engagement_volume'] = df['visit_per_week'] * df['duration_in_gym_minutes']
df['relative_price'] = df['final_price'] / (df['subscription_price'] + 1e-5)

# 6. Dịch vụ có trọng số
df['weighted_services'] = (
    df['has_drink_subscription'].astype(int) * 1 +
    df['personal_training'].astype(int) * 2 +
    df['uses_sauna'].astype(int) * 1 +
    df['multi_location_access'].astype(int) * 2 +
    df['attend_group_lesson'].astype(int) * 1
)

# 11. Giá
df['discount_amount'] = df['subscription_price'] - df['final_price']
df['has_discount']    = (df['discount_rate'] > 0).astype(int)

print(f'Shape: {df.shape}')
print("Các cột mới được thêm:", [c for c in df.columns if c not in df_raw.columns])

Shape: (1998, 39)
Các cột mới được thêm: ['join_season', 'join_month', 'join_year', 'avg_time_check_in_hour', 'avg_time_check_out_hour', 'engagement_volume', 'relative_price', 'weighted_services', 'discount_amount', 'has_discount']


## 3. Train/Test split 

In [6]:
TARGET = 'churn'
X = df.drop(columns=[TARGET])
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} samples | Test: {X_test.shape[0]:,} samples')
print(f'Tỷ lệ churn — train: {y_train.value_counts(normalize=True).to_dict()}')
print(f'Tỷ lệ churn — test : {y_test.value_counts(normalize=True).to_dict()}')

Train: 1,598 samples | Test: 400 samples
Tỷ lệ churn — train: {1: 0.39612015018773467, 0: 0.35294117647058826, 2: 0.2509386733416771}
Tỷ lệ churn — test : {1: 0.3975, 0: 0.3525, 2: 0.25}


In [7]:
X_train.columns

Index(['age', 'membership_type', 'visit_per_week', 'days_per_week',
       'attend_group_lesson', 'avg_time_check_in', 'avg_time_check_out',
       'duration_in_gym_minutes', 'has_drink_subscription',
       'personal_training', 'uses_sauna', 'self_identified_gender',
       'subscription_price', 'subscription_model', 'adjusted_price',
       'discount_type', 'discount_rate', 'final_price', 'access_hours',
       'home_gym_location', 'latitude', 'longitude', 'join_date',
       'personal_training_hours', 'multi_location_access', 'last_visit_date',
       'id', 'days_since_last_visit', 'join_season', 'join_month', 'join_year',
       'avg_time_check_in_hour', 'avg_time_check_out_hour',
       'engagement_volume', 'relative_price', 'weighted_services',
       'discount_amount', 'has_discount'],
      dtype='object')

In [8]:
X_train.shape

(1598, 38)

## 4. Drop cột 

In [9]:
DROP_COLS = [
    'days_since_last_visit',  
    'last_visit_date',
    'join_date',
    'days_per_week',
    'avg_time_check_in',
    'avg_time_check_out',
    'latitude', 'longitude', 'home_gym_location',
    'adjusted_price',
    'id',
    'visit_per_week',
    'subscription_price',
    'discount_type',
    'duration_in_gym_minutes',
]

drop_train = [c for c in DROP_COLS if c in X_train.columns]
drop_test  = [c for c in DROP_COLS if c in X_test.columns]

X_train = X_train.drop(columns=drop_train)
X_test  = X_test.drop(columns=drop_test)

print(f'Đã drop {len(drop_train)} cột')
print(f'Shape — train: {X_train.shape} | test: {X_test.shape}')
print(f'\nCột còn lại ({len(X_train.columns)}):')
for c in X_train.columns:
    print(f'  {c}')

Đã drop 15 cột
Shape — train: (1598, 23) | test: (400, 23)

Cột còn lại (23):
  age
  membership_type
  attend_group_lesson
  has_drink_subscription
  personal_training
  uses_sauna
  self_identified_gender
  subscription_model
  discount_rate
  final_price
  access_hours
  personal_training_hours
  multi_location_access
  join_season
  join_month
  join_year
  avg_time_check_in_hour
  avg_time_check_out_hour
  engagement_volume
  relative_price
  weighted_services
  discount_amount
  has_discount


## 5. Encoding dữ liệu

In [10]:
bool_cols = X_train.select_dtypes(include='bool').columns.tolist()
cat_cols  = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols  = X_train.select_dtypes(include='number').columns.tolist()

print(f'Bool : {bool_cols}')
print(f'Cat  : {cat_cols}')
print(f'Num  : {num_cols}')

Bool : ['attend_group_lesson', 'has_drink_subscription', 'personal_training', 'uses_sauna', 'multi_location_access']
Cat  : ['membership_type', 'self_identified_gender', 'subscription_model', 'access_hours', 'join_season']
Num  : ['age', 'discount_rate', 'final_price', 'personal_training_hours', 'join_month', 'join_year', 'avg_time_check_in_hour', 'avg_time_check_out_hour', 'engagement_volume', 'relative_price', 'weighted_services', 'discount_amount', 'has_discount']


In [11]:
# Bool → int
for col in bool_cols:
    X_train[col] = X_train[col].astype(int)
    X_test[col]  = X_test[col].astype(int)

In [ ]:
encoders = {}

cat_cols  = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    n_unique = X_train[col].nunique()

    le = LabelEncoder()
    le.fit(X_train[col].astype(str))
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col]  = X_test[col].astype(str).apply(
        lambda v: le.transform([v])[0] if v in le.classes_ else -1
    )


## 6. Scaling dữ liệu

In [ ]:
binary_cols = [c for c in X_train.columns if X_train[c].nunique() <= 2]
scale_cols  = [c for c in X_train.select_dtypes(include='number').columns
               if c not in binary_cols]

scaler = StandardScaler()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])  
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])       


## 7. Train model — Cross-validation 5-fold

In [14]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost'      : XGBClassifier(random_state=42,)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    t0 = time.time()
    acc_s, f1_s, prec_s, rec_s, auc_s = [], [], [], [], []

    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        m = clone(model)
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_val)

        acc_s.append(accuracy_score(y_val, y_pred))
        f1_s.append(f1_score(y_val, y_pred, average='macro'))
        prec_s.append(precision_score(y_val, y_pred, average='macro', zero_division=0))
        rec_s.append(recall_score(y_val, y_pred, average='macro', zero_division=0))
        if hasattr(m, 'predict_proba'):
            auc_s.append(roc_auc_score(y_val, m.predict_proba(X_val),
                                       multi_class='ovr', average='macro'))
        
    cv_results[name] = {
        'acc_mean': np.mean(acc_s), 'acc_std': np.std(acc_s),
        'prec_mean': np.mean(prec_s), 'prec_std': np.std(prec_s),
        'rec_mean': np.mean(rec_s), 'rec_std': np.std(rec_s),
        'f1_mean': np.mean(f1_s), 'f1_std': np.std(f1_s),
        'auc_mean': np.mean(auc_s), 'auc_std': np.std(auc_s),
    }

    r = cv_results[name]
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(f"  Accuracy : {r['acc_mean']:.4f} ± {r['acc_std']:.4f}")
    print(f"  Precision: {r['prec_mean']:.4f} ± {r['prec_std']:.4f}")
    print(f"  Recall   : {r['rec_mean']:.4f} ± {r['rec_std']:.4f}")
    print(f"  F1-score : {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")
    print(f"  AUC      : {r['auc_mean']:.4f} ± {r['auc_std']:.4f}")
    


Logistic Regression
  Accuracy : 0.9725 ± 0.0096
  Precision: 0.9736 ± 0.0092
  Recall   : 0.9717 ± 0.0098
  F1-score : 0.9725 ± 0.0094
  AUC      : 0.9989 ± 0.0006

Random Forest
  Accuracy : 0.9643 ± 0.0149
  Precision: 0.9674 ± 0.0141
  Recall   : 0.9627 ± 0.0136
  F1-score : 0.9647 ± 0.0137
  AUC      : 0.9926 ± 0.0035

XGBoost
  Accuracy : 0.9681 ± 0.0093
  Precision: 0.9691 ± 0.0100
  Recall   : 0.9678 ± 0.0094
  F1-score : 0.9682 ± 0.0094
  AUC      : 0.9991 ± 0.0005


In [15]:
summary = pd.DataFrame([
    {'Model'   : name,
     'Accuracy': f"{r['acc_mean']:.4f} ± {r['acc_std']:.4f}",
     'Precision': f"{r['prec_mean']:.4f} ± {r['prec_std']:.4f}",
     'Recall'  : f"{r['rec_mean']:.4f} ± {r['rec_std']:.4f}",
     'F1-macro': f"{r['f1_mean']:.4f} ± {r['f1_std']:.4f}",
     'AUC'     : f"{r['auc_mean']:.4f} ± {r['auc_std']:.4f}"}
    for name, r in cv_results.items()
])
display(summary)

,Model,Accuracy,Precision,Recall,F1-macro,AUC
0,Logistic Regression,0.9725 ± 0.0096,0.9736 ± 0.0092,0.9717 ± 0.0098,0.9725 ± 0.0094,0.9989 ± 0.0006
1,Random Forest,0.9643 ± 0.0149,0.9674 ± 0.0141,0.9627 ± 0.0136,0.9647 ± 0.0137,0.9926 ± 0.0035
2,XGBoost,0.9681 ± 0.0093,0.9691 ± 0.0100,0.9678 ± 0.0094,0.9682 ± 0.0094,0.9991 ± 0.0005


## 8. Đánh giá model tốt nhất trên Test set

In [ ]:
# best_name = max(test_results, key=lambda k: test_results[k]['f1_mean'])
# print(f'Model tốt nhất (F1-macro Test): {best_name}')
test_results = { }
for model_name in models:

    model = clone(models[model_name])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    
    acc_s.append(accuracy_score(y_test, y_pred))
    f1_s.append(f1_score(y_test, y_pred, average='macro'))
    prec_s.append(precision_score(y_test, y_pred, average='macro', zero_division=0))
    rec_s.append(recall_score(y_test, y_pred, average='macro', zero_division=0))
    if hasattr(m, 'predict_proba'):
            auc_s.append(roc_auc_score(y_val, m.predict_proba(X_val),
                                       multi_class='ovr', average='macro'))
        
    test_results[model_name] = {
        'acc_mean': np.mean(acc_s), 'acc_std': np.std(acc_s),
        'prec_mean': np.mean(prec_s), 'prec_std': np.std(prec_s),
        'rec_mean': np.mean(rec_s), 'rec_std': np.std(rec_s),
        'f1_mean': np.mean(f1_s), 'f1_std': np.std(f1_s),
        'auc_mean': np.mean(auc_s), 'auc_std': np.std(auc_s),
    }

    r = test_results[model_name]
    print(f"\n{'='*50}\n{model_name}\n{'='*50}")
    print(f"  Accuracy : {r['acc_mean']:.4f} ± {r['acc_std']:.4f}")
    print(f"  Precision: {r['prec_mean']:.4f} ± {r['prec_std']:.4f}")
    print(f"  Recall   : {r['rec_mean']:.4f} ± {r['rec_std']:.4f}")
    print(f"  F1-score : {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")
    print(f"  AUC      : {r['auc_mean']:.4f} ± {r['auc_std']:.4f}")
    



Logistic Regression
  Accuracy : 0.9676 ± 0.0086
  Precision: 0.9685 ± 0.0093
  Recall   : 0.9677 ± 0.0086
  F1-score : 0.9678 ± 0.0086
  AUC      : 0.9990 ± 0.0005

Random Forest
  Accuracy : 0.9676 ± 0.0080
  Precision: 0.9686 ± 0.0086
  Recall   : 0.9676 ± 0.0080
  F1-score : 0.9679 ± 0.0080
  AUC      : 0.9989 ± 0.0005

XGBoost
  Accuracy : 0.9672 ± 0.0075
  Precision: 0.9682 ± 0.0081
  Recall   : 0.9673 ± 0.0075
  F1-score : 0.9676 ± 0.0075
  AUC      : 0.9989 ± 0.0005


## 9. Tunning model

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score, classification_report
import time
import pandas as pd

# Định nghĩa scorer
f1_scorer = make_scorer(f1_score, average='macro')

# Khởi tạo các mô hình (không pipeline)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, multi_class='ovr'),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0),
}

# Lưới tham số - bỏ tiền tố classifier__ vì không còn pipeline
param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10, 100],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'newton-cg']
    },
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2']
    },
    'XGBoost': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 6, 9],
        'subsample': [0.7, 0.8, 1.0],
        'colsample_bytree': [0.7, 0.8, 1.0]
    }
}

# Tuning
tuning_results = {}
for name in models.keys():
    print(f"\n--- Tuning cho {name} ---")
    t0 = time.time()
    search = GridSearchCV(
        estimator=models[name],
        param_grid=param_grids[name],
        scoring=f1_scorer,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_train, y_train)   # LƯU Ý: NẾU DÙNG LOGISTIC, CẦN ĐẢM BẢO X_train ĐÃ ĐƯỢC SCALE
    tuning_results[name] = {
        'best_params': search.best_params_,
        'best_cv_f1': search.best_score_,
        'best_estimator': search.best_estimator_,
        'time': time.time() - t0
    }
    print(f"  Best CV F1-macro: {search.best_score_:.4f}")
    print(f"  Time: {tuning_results[name]['time']:.2f}s")
from sklearn.metrics import roc_auc_score  # thêm dòng này nếu chưa có

# ================= ĐÁNH GIÁ SAU TUNING TRÊN TẬP TEST =================
print("\n" + "="*60)
print("ĐÁNH GIÁ SAU TUNING TRÊN TẬP TEST")
print("="*60)

test_results = {}
for name, res in tuning_results.items():
    model = res['best_estimator']
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    # Tính AUC nếu model có predict_proba
    if hasattr(model, 'predict_proba'):
        auc = roc_auc_score(y_test, model.predict_proba(X_test),
                            multi_class='ovr', average='macro')
    else:
        auc = None
    
    test_results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-macro': f1,
        'AUC': auc
    }
    print(f"\n{name} (sau tuning):")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-macro : {f1:.4f}")
    if auc is not None:
        print(f"  AUC      : {auc:.4f}")

# Tạo DataFrame kết quả
test_df = pd.DataFrame([
    {'Model': name,
     'Accuracy': f"{res['Accuracy']:.4f}",
     'Precision': f"{res['Precision']:.4f}",
     'Recall': f"{res['Recall']:.4f}",
     'F1-macro': f"{res['F1-macro']:.4f}",
     'AUC': f"{res['AUC']:.4f}" if res['AUC'] is not None else 'N/A'}
    for name, res in test_results.items()
])
print("\n===== KẾT QUẢ TRÊN TẬP TEST (SAU TUNING) =====")
display(test_df)

# Chọn mô hình tốt nhất dựa trên CV F1-macro
best_tuned_name = max(tuning_results, key=lambda k: tuning_results[k]['best_cv_f1'])
best_tuned_model = tuning_results[best_tuned_name]['best_estimator']
print(f"\n🏆 Mô hình tốt nhất sau tuning: {best_tuned_name} (CV F1-macro = {tuning_results[best_tuned_name]['best_cv_f1']:.4f})")

# Đánh giá cuối cùng trên test
y_pred_final = best_tuned_model.predict(X_test)
print("\n=== ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"F1-macro  : {f1_score(y_test, y_pred_final, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, target_names=['Đang dùng (0)', 'Có nguy cơ (1)', 'Đã rời bỏ (2)']))


--- Tuning cho Logistic Regression ---
Fitting 5 folds for each of 10 candidates, totalling 50 fits
  Best CV F1-macro: 0.9726
  Time: 8.66s

--- Tuning cho Random Forest ---
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
  Best CV F1-macro: 0.9690
  Time: 217.21s

--- Tuning cho XGBoost ---
Fitting 5 folds for each of 324 candidates, totalling 1620 fits
  Best CV F1-macro: 0.9737
  Time: 412.16s

ĐÁNH GIÁ SAU TUNING TRÊN TẬP TEST

Logistic Regression (sau tuning):
  Accuracy : 0.9650
  Precision: 0.9659
  Recall   : 0.9659
  F1-macro : 0.9659
  AUC      : 0.9952

Random Forest (sau tuning):
  Accuracy : 0.9750
  Precision: 0.9771
  Recall   : 0.9736
  F1-macro : 0.9752
  AUC      : 0.9969

XGBoost (sau tuning):
  Accuracy : 0.9700
  Precision: 0.9713
  Recall   : 0.9703
  F1-macro : 0.9708
  AUC      : 0.9989

===== KẾT QUẢ TRÊN TẬP TEST (SAU TUNING) =====


,Model,Accuracy,Precision,Recall,F1-macro,AUC
0,Logistic Regression,0.9650,0.9659,0.9659,0.9659,0.9952
1,Random Forest,0.9750,0.9771,0.9736,0.9752,0.9969
2,XGBoost,0.9700,0.9713,0.9703,0.9708,0.9989



🏆 Mô hình tốt nhất sau tuning: XGBoost (CV F1-macro = 0.9737)

=== ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST ===
Accuracy  : 0.9700
F1-macro  : 0.9708

Classification Report:
                precision    recall  f1-score   support

 Đang dùng (0)       0.97      0.98      0.98       141
Có nguy cơ (1)       0.96      0.96      0.96       159
 Đã rời bỏ (2)       0.98      0.97      0.97       100

      accuracy                           0.97       400
     macro avg       0.97      0.97      0.97       400
  weighted avg       0.97      0.97      0.97       400



In [ ]:
summary = pd.DataFrame([
    {'Model'   : name,
     'Accuracy': f"{r['acc_mean']:.4f} ± {r['acc_std']:.4f}",
     'Precision': f"{r['prec_mean']:.4f} ± {r['prec_std']:.4f}",
     'Recall'  : f"{r['rec_mean']:.4f} ± {r['rec_std']:.4f}",
     'F1-macro': f"{r['f1_mean']:.4f} ± {r['f1_std']:.4f}",
     'AUC': f"{r['auc_mean']:.4f} ± {r['auc_std']:.4f}"}
    for name, r in test_results.items()
])
display(summary)

,Model,Accuracy,Precision,Recall,F1-macro,AUC
0,Logistic Regression,0.9672 ± 0.0080,0.9680 ± 0.0087,0.9676 ± 0.0080,0.9676 ± 0.0080,0.9990 ± 0.0005
1,Random Forest,0.9672 ± 0.0075,0.9681 ± 0.0081,0.9675 ± 0.0074,0.9676 ± 0.0075,0.9989 ± 0.0005
2,XGBoost,0.9670 ± 0.0071,0.9678 ± 0.0077,0.9673 ± 0.0070,0.9674 ± 0.0071,0.9989 ± 0.0005


In [22]:
print("\n" + "="*60)
print("THAM SỐ TỐI ƯU CỦA TỪNG MÔ HÌNH")
print("="*60)
for name, res in tuning_results.items():
    print(f"\n{name}:")
    for param, value in res['best_params'].items():
        print(f"    {param}: {value}")


THAM SỐ TỐI ƯU CỦA TỪNG MÔ HÌNH

Logistic Regression:
    C: 100
    penalty: l2
    solver: lbfgs

Random Forest:
    max_depth: None
    max_features: sqrt
    min_samples_leaf: 1
    min_samples_split: 5
    n_estimators: 200

XGBoost:
    colsample_bytree: 0.7
    learning_rate: 0.2
    max_depth: 9
    n_estimators: 100
    subsample: 0.7
